In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/dataset"


In [ ]:
!pip install opencv-python scikit-image


In [ ]:
import cv2
import os
import numpy as np
from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib

IMG_SIZE = 128
DATASET_PATH = "/content/drive/MyDrive/dataset"

# -------------------------------
# Feature Extraction
# -------------------------------
def extract_features(segmented):

    # Calculate lesion area (number of white pixels)
    area = cv2.countNonZero(segmented)

    # Create GLCM matrix for texture analysis
    glcm = graycomatrix(segmented, [1], [0], 256, symmetric=True, normed=True)

    # Extract contrast feature (texture roughness)
    contrast = graycoprops(glcm, 'contrast')[0, 0]

    # Extract energy feature (texture uniformity)
    energy = graycoprops(glcm, 'energy')[0, 0]

    return [area, contrast, energy]

# -------------------------------
# Load Dataset
# -------------------------------
features = []
labels = []

for label, folder in enumerate(["benign", "malignant"]):
    folder_path = os.path.join(DATASET_PATH, folder)

    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)

        img = cv2.imread(img_path) # Read image
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) # Resize image

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) # Convert to grayscale
        blur = cv2.GaussianBlur(gray, (5, 5), 0) # Apply Gaussian blur to remove noise

# Apply Otsu thresholding for segmentation
        _, segmented = cv2.threshold(
            blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

 # Ensure lesion is white and background is black
        if np.sum(segmented == 255) > np.sum(segmented == 0):
            segmented = cv2.bitwise_not(segmented)

        features.append(extract_features(segmented))
        labels.append(label)

X = np.array(features)
y = np.array(labels)

# -------------------------------
# Train-Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------
# Train SVM
# -------------------------------
model = SVC(kernel='linear')
model.fit(X_train, y_train)

# -------------------------------
# Evaluation
# -------------------------------
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.6341463414634146
              precision    recall  f1-score   support

           0       0.73      0.50      0.59        22
           1       0.58      0.79      0.67        19

    accuracy                           0.63        41
   macro avg       0.66      0.64      0.63        41
weighted avg       0.66      0.63      0.63        41



In [ ]:
joblib.dump(model, "/content/drive/MyDrive/skin_cancer_svm.pkl")
print("✅ Model saved to Google Drive")


✅ Model saved to Google Drive


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import joblib

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = SVC(kernel='rbf', class_weight='balanced')
model.fit(X_scaled, y)

joblib.dump(model, "/content/drive/MyDrive/skin_cancer_svm.pkl")
joblib.dump(scaler, "/content/drive/MyDrive/skin_cancer_scaler.pkl")

print("✅ Model & scaler saved")


✅ Model & scaler saved


In [ ]:
import gradio as gr
import cv2
import numpy as np
import joblib
from skimage.feature import graycomatrix, graycoprops

IMG_SIZE = 128

# Load trained model & scaler
model = joblib.load("/content/drive/MyDrive/skin_cancer_svm.pkl")
scaler = joblib.load("/content/drive/MyDrive/skin_cancer_scaler.pkl")

def extract_features(segmented):
    area = cv2.countNonZero(segmented)

    glcm = graycomatrix(segmented, [1], [0], 256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]

    return [area, contrast, energy]

def dashboard(image):
    if image is None:
        return "Upload image", None, None

    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    _, segmented = cv2.threshold(
        blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Ensure lesion is white
    if np.sum(segmented == 255) > np.sum(segmented == 0):
        segmented = cv2.bitwise_not(segmented)

    features = np.array(extract_features(segmented)).reshape(1, -1)

    # SCALE FEATURES (CRITICAL)
    features = scaler.transform(features)

    prediction = model.predict(features)[0]
    result = "Benign" if prediction == 0 else "Malignant"

    return result, gray, segmented

gr.Interface(
    fn=dashboard,
    inputs=gr.Image(type="numpy"),
    outputs=[
        gr.Textbox(label="Diagnosis"),
        gr.Image(label="Preprocessed Image"),
        gr.Image(label="Segmented Image")
    ],
    title="Skin Cancer Detection using SVM",
    description="GLCM + Lesion Area + SVM Classification"
).launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ac705e815afe05dc8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
